# 강의 04 · 실습 8 — 이미지 생성 파이프라인 · (2-1) 빈칸 채우기 I

## 1. 문제상황

- 운영팀 디자이너는 홍보용 이미지를 이미지 생성 모델로 만듭니다.
- 주제를 받으면 디자이너가 지시문을 손으로 쓰고, 모델을 불러 그림을 뽑고, 결과를 눈으로 봅니다.
- 그림이 마음에 들지 않으면 지시문을 고쳐 다시 뽑는 일을 손으로 되풀이합니다.
- 주제가 늘어나면 지시문 작성과 되풀이를 사람이 그만큼 반복해야 합니다.

## 2. 문제와 목표

- **문제**: 지시문 작성과 다시 뽑기를 사람이 손으로 되풀이합니다. 사람이 해야 할 일은 결과를 보고 고르는 것뿐인데, 지시문 쓰기와 호출까지 사람이 맡고 있습니다.
- **목표**: 주제를 입력하면 프로그램이 지시문을 쓰고 이미지를 뽑고, 사람은 후보를 보고 「확정」 또는 「재설계」만 답하며, 재설계이면 지시문 설계부터 다시 도는 처리 흐름을 만듭니다.
    - 상태 키 네 개: 주제, 지시문, 생성한 이미지 경로 목록(뒤에 이어 붙음), 사람의 판정입니다.
    - 노드 세 개: 지시문을 쓰는 design(언어 모델), 이미지를 뽑는 generate(이미지 모델), 멈추고 사람의 답을 받는 review입니다.
    - 사람의 답(「재설계」 한 번, 「확정」 한 번)은 코드에 대본으로 미리 정해 넣습니다. 실제 서비스에서는 사람이 화면에서 입력합니다.
- **목표 달성 여부의 판정 기준**
    - 주제 하나를 입력했을 때 그래프가 사람의 판정을 기다리며 멈춥니다.
    - 「재설계」라고 답하면 지시문 설계와 이미지 생성이 다시 실행되어 후보가 2장으로 늡니다.
    - 「확정」이라고 답하면 END에 도달하는 것을 실행 결과에서 확인합니다.


## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec04_ex08_s1_diagram.svg)

## 4. 단계별 요구사항

1. **상태를 정의합니다.**
    - 주제(`topic`), 이미지 지시문(`prompt`), 생성한 이미지 경로 목록(`images`), 사람의 판정(`verdict`) 키 네 개를 가지는 상태를 선언합니다.
    - `images` 키에는 `add` 리듀서를 붙여, 노드가 돌려준 목록이 기존 목록 뒤에 이어 붙게 합니다.
2. **지시문 설계 노드를 만듭니다.**
    - design 노드는 상태의 주제를 지시문 템플릿 `SPEC_V1` 뒤에 붙여 모델을 한 번 호출하고, 받은 지시문 한 문단을 `prompt` 키에 씁니다.
3. **이미지 생성 노드를 만듭니다.**
    - generate 노드는 상태의 지시문을 `paint`로 이미지 모델에 보내 파일로 저장하고, 저장된 경로 하나를 담은 목록을 `images` 키에 돌려줍니다.
4. **평가 노드를 만듭니다.**
    - review 노드는 `interrupt()`로 실행을 멈추고, 질문과 지금까지의 후보 경로 목록을 사람에게 보냅니다.
    - 사람의 답을 문자열로 `verdict` 키에 씁니다.
5. **그래프에 노드를 등록합니다.**
    - 세 노드를 이름과 함께 그래프에 등록합니다.
6. **엣지를 연결합니다.**
    - START → design → generate → review를 고정 엣지로 연결하고, review 뒤에는 판정을 보고 갈 곳을 고르는 조건부 엣지를 추가합니다.
    - 판정이 「재설계」이면 design으로 되돌아가고, 그 밖에는 END로 갑니다.
7. **그래프를 컴파일하고 실행합니다.**
    - `interrupt()`가 동작하도록 체크포인터(`MemorySaver`)를 달아 컴파일하고, `thread_id`를 담은 설정을 만듭니다.
    - 주제 하나를 넣어 멈춘 지점과 사람에게 간 내용을 출력하고 후보 이미지를 화면에 표시합니다.
    - `Command(resume="재설계")`로 한 번 되돌린 뒤 후보가 늘어난 것을 확인하고, `Command(resume="확정")`으로 종료합니다.

## 5. 코드 골격 — LangGraph 5단

랭그래프(LangGraph)로 그래프를 세우는 순서는 다음 다섯 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 다섯 단계와 하나씩 대응합니다. 이미지 모델을 부르는 노드와 `interrupt()`는 새 단계가 아니라 ② 노드 함수와 ⑤ 실행 단계 안에 들어갑니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 상태 정의 | 노드들이 함께 읽고 쓸 키를 선언합니다 | `class ImageState(TypedDict)`, `Annotated[list, add]` | 1 |
| ② 노드 함수 정의 | 상태를 받아 바뀐 키만 돌려주는 함수를 만듭니다 | `def design(state) -> dict`, `paint()`, `interrupt()` | 2, 3, 4 |
| ③ 그래프 빌더 생성과 노드 등록 | 빈 그래프를 열고 함수에 이름을 붙여 등록합니다 | `StateGraph(ImageState)`, `add_node` | 5 |
| ④ 엣지 연결 | 노드 사이의 순서와 분기를 정합니다 | `add_edge`, `add_conditional_edges` | 6 |
| ⑤ 컴파일과 실행 | 체크포인터를 달아 컴파일하고, 멈춘 지점에서 사람의 답으로 이어 갑니다 | `compile(checkpointer=…)`, `invoke`, `Command(resume=…)` | 7 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 텍스트 모델과 이미지 모델을 준비합니다. 이미지 모델을 부르는 함수 `paint`도 여기서 정의합니다.

- API 키와 자격증명은 `.env` 파일에서 읽습니다. `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다.
- `.env` 파일에는 다음 네 줄이 있어야 합니다. 값은 각자 발급받은 것을 넣습니다.

```
OPENAI_API_KEY=발급받은_키
GOOGLE_APPLICATION_CREDENTIALS=서비스_계정_키_파일의_경로
VERTEX_PROJECT=프로젝트_이름
VERTEX_LOCATION=리전_이름
```

- `paint` 호출 1회가 이미지 1장이고, 호출마다 비용이 듭니다. 생성한 이미지는 노트북 옆의 `out_images` 폴더에 저장됩니다.
- `show`는 후보 이미지 경로 목록을 화면에 표시하는 보조 함수입니다.

In [ ]:
import os
import time
from operator import add
from pathlib import Path

from dotenv import ____, ____
from typing import ____, ____

from IPython.display import Image, display
from google import genai
from google.genai import types
from langchain.chat_models import ____
from langgraph.checkpoint.memory import ____
from langgraph.graph import ____, ____, ____
from langgraph.types import ____, ____

load_dotenv(find_dotenv(usecwd=True))
for key in ("OPENAI_API_KEY", "GOOGLE_APPLICATION_CREDENTIALS", "VERTEX_PROJECT", "VERTEX_LOCATION"):
    if not os.environ.get(key):
        raise SystemExit(f"agentic-ai 폴더의 .env 파일에 {key} 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
client = genai.Client(vertexai=True, project=os.environ["VERTEX_PROJECT"], location=os.environ["VERTEX_LOCATION"])
IMG_MODEL = "gemini-3.1-flash-lite-image"
OUT_DIR = Path("out_images")
OUT_DIR.mkdir(exist_ok=True)
paint_calls = 0


def paint(prompt: str, tag: str) -> str:
    """지시문을 이미지 모델에 보내 이미지 파일을 저장하고 경로를 돌려준다. 호출 1회 = 이미지 1장 = 비용 발생."""
    global paint_calls
    for attempt in range(4):
        try:
            res = client.models.generate_content(
                model=IMG_MODEL, contents=prompt,
                config=types.GenerateContentConfig(response_modalities=["IMAGE", "TEXT"]))
            break
        except Exception as e:
            if "429" in str(e) and attempt < 3:
                print(f"    [paint] 분당 한도 초과 — {30 * (attempt + 1)}초 뒤 다시 부릅니다")
                time.sleep(30 * (attempt + 1))
                continue
            raise
    paint_calls += 1
    part = [p for p in res.candidates[0].content.parts if p.inline_data][0].inline_data
    ext = "png" if "png" in part.mime_type else "jpg"
    path = OUT_DIR / f"{tag}_{paint_calls:02d}.{ext}"
    path.write_bytes(part.data)
    print(f"    [paint] {path.as_posix()} ({len(part.data)} bytes)")
    return path.as_posix()


def show(paths: list) -> None:
    """후보 이미지 경로 목록을 화면에 차례로 표시한다.

    입력: 이미지 경로 목록
    기능: 경로마다 번호와 경로를 출력하고, Image(filename=경로, width=320)를 display로 표시한다.
    반환: 없음
    """
    # 여기에 출력 루프를 작성합니다.


print("모델 준비를 마쳤습니다. 이미지 저장 폴더:", OUT_DIR)

### 단계 ① — 상태 정의 (요구사항 1)

그래프가 도는 동안 모든 노드가 함께 읽고 쓰는 키를 선언합니다. `images` 키의 `add`가 리듀서입니다. 노드가 `images`에 목록을 돌려주면 리듀서가 기존 목록 뒤에 이어 붙입니다. 이미지 자체가 아니라 저장된 파일의 경로가 담깁니다.

In [ ]:
class ImageState(TypedDict):
    """그래프의 모든 노드가 함께 읽고 쓰는 상태.

    키: topic(주제), prompt(이미지 지시문), images(생성한 이미지 경로 목록, add 리듀서), verdict(사람의 판정)
    """
    # 여기에 키 네 개를 이름과 타입으로 선언합니다. images 키에는 리듀서를 붙입니다.


print("상태의 키:", list(ImageState.__annotations__))

### 단계 ② — 노드 함수 정의 (요구사항 2, 3, 4)

- 노드는 상태를 인자로 받아 딕셔너리를 돌려주는 파이썬 함수입니다. 돌려준 딕셔너리가 상태의 해당 키에 반영됩니다.
- `SPEC_V1`은 design 노드가 쓰는 지시문 템플릿입니다. 주제만 바뀌고 나머지 문면은 고정입니다.
- review 노드의 `interrupt()`는 노드 실행을 그 자리에서 멈추고, 사람이 준 값을 그 호출의 반환값으로 받아 이어 가는 함수입니다.

In [ ]:
SPEC_V1 = ("다음 주제로 이미지 생성 지시문을 한 문단으로 쓴다. "
           "장면·조명·화각을 포함한다. 주제: ")
TOPIC = "비 내리는 밤 서울 골목의 LP 바 창가"


def design(state: ImageState) -> dict:
    """주제를 템플릿에 넣어 이미지 지시문 한 문단을 쓴다.

    입력: 상태의 topic 키
    기능: SPEC_V1 뒤에 주제를 붙여 모델을 한 번 호출한다.
    반환: {"prompt": 지시문 한 문단}
    """
    # 여기에 모델 호출을 작성합니다.
    return {"prompt": ____}


def generate(state: ImageState) -> dict:
    """지시문으로 이미지를 뽑아 저장하고, 경로를 목록에 담아 돌려준다.

    입력: 상태의 prompt 키
    기능: paint(지시문, "s1")로 이미지를 저장하고 경로를 받는다.
    반환: {"images": [저장된 경로]}
    """
    # 여기에 paint 호출을 작성합니다.
    return {"images": ____}


def review(state: ImageState) -> dict:
    """멈추고, 지금까지의 후보를 사람에게 보내 판정을 받는다.

    입력: 상태의 images 키
    기능: interrupt()에 질문과 후보 경로 목록을 담아 멈춘다. 돌아온 값을 문자열로 바꾼다.
    반환: {"verdict": 사람의 답}
    """
    # 여기에 interrupt 호출을 작성합니다.
    return {"verdict": ____}

### 단계 ③ — 그래프 빌더 생성과 노드 등록 (요구사항 5)

`StateGraph`에 상태를 넘겨 빈 그래프를 열고, `add_node`로 함수마다 이름을 붙여 등록합니다.

In [ ]:
g = StateGraph(____)
g.add_node(____)
g.add_node(____)
g.add_node(____)

print("등록한 노드:", list(g.nodes))

### 단계 ④ — 엣지 연결 (요구사항 6)

`add_edge`는 고정된 순서로 연결합니다. `add_conditional_edges`는 판단 함수 `route`가 돌려준 이름으로 다음 노드가 나뉘는 분기를 추가합니다. 되돌림은 그래프에 그어진 엣지이며, 별도의 반복문이 아닙니다.

In [ ]:
def route(state: ImageState) -> str:
    """다음에 갈 노드의 이름을 돌려준다.

    입력: 상태의 verdict 키
    기능: 판정이 「재설계」인지 아닌지를 본다.
    반환: 재설계이면 "design", 아니면 END
    """
    # 여기에 분기 조건을 작성합니다.
    return ____


g.add_edge(____)
g.add_edge(____)
g.add_edge(____)
g.add_conditional_edges(____)

print("고정 엣지 세 개와 조건부 엣지 하나를 놓았습니다.")

### 단계 ⑤ — 컴파일과 실행 (요구사항 7)

`interrupt()`는 저장된 상태 위에서 멈추므로 체크포인터가 필요합니다. 이 실습은 프로그램을 다시 켤 일이 없으므로 메모리 체크포인터(`MemorySaver`)를 씁니다. 아래 세 셀(⑤-a ~ ⑤-c)이 모두 이 한 단계에 속합니다.

In [ ]:
# 단계 ⑤에는 골격이 없습니다. 「4. 단계별 요구사항」의 7번을 보고 처음부터 작성합니다.
# 여기에 체크포인터를 단 컴파일, thread_id 설정, 1차 실행(멈춤 확인·후보 표시), 재설계 재개, 확정 재개를 작성합니다.

## 7. 실행 결과 확인

위 실행 결과에서 다음 네 가지를 확인합니다.

1. 1차 실행에서 `__interrupt__ 있음 = True`, `next = ('review',)`가 출력되고, 사람에게 간 내용의 후보 목록에 경로 1개가 들어 있으며 그 이미지가 화면에 표시됩니다.
2. 재개 A에서 design이 다시 실행되어 새 지시문이 출력되고, 후보가 2장으로 늘어난 채 review에서 다시 멈춥니다. 앞 실행의 후보가 지워지지 않고 남은 것은 `images` 키의 `add` 리듀서 때문입니다.
3. 재개 B에서 `verdict = 확정`, `next = ()`가 출력됩니다. 빈 튜플은 END에 도달했다는 뜻입니다. 생성 호출은 2회입니다.
4. 「6. 코드 — 스텝바이스텝」의 코드에는 반복문이 없습니다. 되돌림은 `add_conditional_edges`로 그어진 엣지이며, route가 돌려준 이름이 다음 노드가 됩니다.

확인이 하나라도 다르면 `lec04_ex08_s1.ipynb`와 대조해 채운 빈칸을 고칩니다.
